In [1]:
from clickhouse_driver import Client

In [17]:
CLICKHOUSE_HOST = 'clickhouse'
CLICKHOUSE_PORT = 9000
CLICKHOUSE_USER = 'admin'
CLICKHOUSE_PASSWORD = 'admin'
CLICKHOUSE_DATABASE = 'dwh'

In [18]:
ch = Client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    user=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

In [19]:
query = """
    SELECT name, value FROM system.settings;
"""

In [20]:
res = ch.execute(query)

In [23]:
res[:5]

[('dialect', 'clickhouse'),
 ('min_compress_block_size', '65536'),
 ('max_compress_block_size', '1048576'),
 ('max_block_size', '65409'),
 ('max_insert_block_size', '1048449')]

Connect to iceberg

In [77]:
ch.execute("SET allow_experimental_database_iceberg = 1;")

[]

In [78]:
ch.execute("""
DROP DATABASE IF EXISTS iceberg;
""")

ch.execute("""
CREATE DATABASE iceberg
ENGINE = DataLakeCatalog('http://nessie-catalog:19120/iceberg', 'ozone-access-key', 'ozone-secret-key')
SETTINGS catalog_type = 'rest', storage_endpoint = 'http://ozone-s3g:9878/mybucket', warehouse = 'warehouse'
;
""")

[]

In [54]:
tables = ch.execute("""
    SHOW tables IN iceberg;
""")

In [55]:
tables

[('cryptocurrencies_project_dds.trade_data',),
 ('cryptocurrencies_project_dds_stg.trade_data',),
 ('cryptocurrencies_project_dm.analytic_indicators',),
 ('cryptocurrencies_project_dm_stg.analytic_indicators',),
 ('cryptocurrencies_project_raw.btc',),
 ('cryptocurrencies_project_raw.eth',),
 ('cryptocurrencies_project_raw_stg.btc',),
 ('cryptocurrencies_project_raw_stg.eth',),
 ('test_iceberg_db.test_iceberg_table',),
 ('test_iceberg_db_2.test_iceberg_table',)]

In [72]:
ch.execute("""
    --SELECT *
    --FROM iceberg.`test_iceberg_db_2.test_iceberg_table`;
    --SELECT * FROM system.databases WHERE name = 'iceberg';
    SELECT *
    FROM icebergS3(
        'http://ozone-s3g:9878/mybucket/warehouse/test_iceberg_db_2/test_iceberg_table_5f0e85b0-f151-477a-bbd6-2bd2528af9f7',
        'ozone-access-key',
        'ozone-secret-key'
    );
""")

[(1115, 'spark job test', '2025-12-31'),
 (1115, 'spark job test', '2025-12-31'),
 (1115, 'spark job test', '2025-12-31'),
 (1, 'tgete', '2025-02-01'),
 (3, 'vdsv', '2025-06-01'),
 (3, 'vdsv', '2025-06-01'),
 (1, 'tgete', '2025-02-01')]

In [16]:
import requests

url = "http://clickhouse:8123/"
auth = ('admin', 'admin')          # логин, пароль
params = {'database': 'dwh'}

query = "SELECT name, value FROM system.settings FORMAT JSON;"

response = requests.post(url, auth=auth, params=params, data=query)

if response.status_code == 200:
    print(response)
    data = response.json()
    for row in data:
        print(row)
else:
    print(f"Ошибка: {response.status_code} {response.text}")

<Response [200]>
meta
data
rows
statistics


In [48]:
import requests
import json

url = "http://nessie-catalog:19120/api/v1/contents/test_iceberg_db_2.test_iceberg_table"
params = {"ref": "main"}
auth = ("ozone-access-key", "ozone-secret-key")  # если нужна авторизация

resp = requests.get(url, params=params, auth=auth)
if resp.status_code == 200:
    data = resp.json()
    print("Metadata location:", data.get("metadataLocation"))
else:
    print(f"Error {resp.status_code}: {resp.text}")

Metadata location: s3a://mybucket/warehouse/test_iceberg_db_2/test_iceberg_table_5f0e85b0-f151-477a-bbd6-2bd2528af9f7/metadata/00005-f87813b6-8a8f-40e5-a753-f248cb8aad37.metadata.json
